# Baseline results


In [1]:
import numpy as np
import pandas as pd

sizes = [30, 50, 100]
feats = ["log_pay", "log_benes", "srvcs_per_bene", "markup"]
rz_cols = [f + "_rz" for f in feats]

feat = pd.read_parquet("data/processed/features.parquet")
peers = {m: pd.read_parquet(f"data/processed/peers_{m}.parquet") for m in sizes}
t = feat["Rndrng_Prvdr_Type"]  # provider type, same row order as the peers files
len(feat)

9781161

In [2]:
def flag_rule(p, one=10, two=3.5, n_min=2):
    # flag = 1 feature > one  OR  n_min+ features > two  (high side only)
    high = p[rz_cols].clip(lower=0).fillna(0)
    one_big = high.max(axis=1) > one
    many = (high > two).sum(axis=1) >= n_min
    return (one_big | many) & (p["peer_level"] > 0)

def jaccard(a, b):
    # % overlap of two sets of flags: both / either
    return (a & b).sum() / (a | b).sum() * 100

flags = {m: flag_rule(peers[m]) for m in sizes}

## 1. Group size

Flag rate overall and by peer level

In [3]:
rows = []
for m in sizes:
    p, f = peers[m], flags[m]
    row = {"min size": m, "flag %": f.mean() * 100}
    for lvl in [1, 2, 3]:
        row[f"flag % on level {lvl}"] = f[p["peer_level"] == lvl].mean() * 100  # flag rate among rows on this level
    rows.append(row)
pd.DataFrame(rows).set_index("min size").round(2)

,flag %,flag % on level 1,flag % on level 2,flag % on level 3
min size,,,,
30,1.21,1.16,3.35,12.49
50,1.22,1.15,3.26,13.09
100,1.24,1.13,2.98,13.69


In [4]:
overlap = pd.DataFrame({a: {b: jaccard(flags[a], flags[b]) for b in sizes} for a in sizes}).round(1)
overlap  # % of flags shared between min sizes (Jaccard)

,30,50,100
30,100.0,95.6,88.9
50,95.6,100.0,92.5
100,88.9,92.5,100.0


In [5]:
# rows on level 1 at min 30 that fall back to a broader group at min 100
moved = (peers[30]["peer_level"] == 1) & (peers[100]["peer_level"] > 1)
only100 = moved & flags[100] & ~flags[30]
print(f"moved rows: {moved.sum():,}, flag % at 30: {flags[30][moved].mean() * 100:.2f}, at 100: {flags[100][moved].mean() * 100:.2f}")

by_type = pd.DataFrame({"moved rows": t[moved].value_counts(),
                        "flagged only at 100 (%)": t[only100].value_counts() / t[moved].value_counts() * 100})
by_type[by_type["moved rows"] >= 1000].sort_values("flagged only at 100 (%)", ascending=False).head(6).round(1)

moved rows: 271,487, flag % at 30: 2.01, at 100: 3.22


,moved rows,flagged only at 100 (%)
Rndrng_Prvdr_Type,,
Ambulatory Surgical Center,5016,36.3
Portable X-Ray Supplier,1593,11.5
Geriatric Medicine,2326,7.0
Diagnostic Radiology,9356,5.5
Thoracic Surgery,1594,5.3
Radiation Oncology,1954,4.7


In [6]:
# which feature is above 3.5 in the flags that appear only at min 100
(peers[100].loc[only100, rz_cols] > 3.5).mean().mul(100).round(1)

log_pay_rz           56.5
log_benes_rz          8.1
srvcs_per_bene_rz    40.2
markup_rz            11.9
dtype: float64

## 2. Thresholds (min size 100)

In [7]:
p = peers[100]
grid = []
for one in [6, 8, 10, 12, 15]:
    for two in [3, 3.5, 4]:
        grid.append({"1 feature >": one, "2+ features >": two, "flag %": flag_rule(p, one=one, two=two).mean() * 100})
grid = pd.DataFrame(grid)
grid.pivot(index="1 feature >", columns="2+ features >", values="flag %").round(2)

2+ features >,3.0,3.5,4.0
1 feature >,,,
6,2.41,2.31,2.26
8,1.72,1.60,1.54
10,1.37,1.24,1.18
12,1.17,1.03,0.97
15,0.99,0.85,0.79


Rows: cut-off for one feature. Columns: cut-off for 2+ features. Cells: % of rows flagged. The number drops a lot going down (one-feature cut-off) and barely changes going right (2+ cut-off).

### Which part of the rule creates the flags (% of flags)

In [8]:
default = flags[100]
high = p[rz_cols].clip(lower=0).fillna(0)
part_one = high.max(axis=1) > 10
part_two = (high > 3.5).sum(axis=1) >= 2
pd.Series({"only 1 feature > 10": (default & part_one & ~part_two).sum(),
           "only 2+ features > 3.5": (default & part_two & ~part_one).sum(),
           "both": (default & part_one & part_two).sum()}).div(default.sum()).mul(100).round(1)  # % of flags

only 1 feature > 10       86.8
only 2+ features > 3.5     8.6
both                       4.6
dtype: float64

### What is gained or lost around 10


In [9]:
# how many times above the peer median (features are logs, so exp(x - median) = ratio)
X = feat[feats]
levels = {1: ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"],
          2: ["HCPCS_Cd", "Place_Of_Srvc"],
          3: ["HCPCS_Cd"]}
med = pd.DataFrame(np.nan, index=X.index, columns=feats)
for lvl, keys in levels.items():
    m = X.groupby([feat[k] for k in keys], observed=True).transform("median")  # median of the whole group on this level
    rows = p["peer_level"] == lvl
    med.loc[rows] = m.loc[rows].to_numpy()
times = np.exp(X - med)  # 2.0 = twice the peer median

In [ ]:
f8, f12 = flag_rule(p, one=8), flag_rule(p, one=12)
bands = {"added at 8": f8 & ~default,  # new rows if we use 8
         "lost at 12": default & ~f12,  # rows we lose if we use 12
         "kept at 12": f12}    # flagged at every cut-off

out = []
for name, b in bands.items():
    z = p.loc[b, rz_cols].clip(lower=0).fillna(0).to_numpy()
    top = z.argmax(axis=1)  # feature with the highest z in each row
    x_top = times.loc[b].to_numpy()[np.arange(len(top)), top]  # its ratio to the peer median
    for i, f in enumerate(feats):
        x = x_top[top == i]
        out.append({"band": name, "top feature": f, "rows": len(x),
                    "median x peer median": np.median(x) if len(x) else np.nan})
out = pd.DataFrame(out)
display(out.pivot(index="top feature", columns="band", values="rows"))
out.pivot(index="top feature", columns="band", values="median x peer median").round(2)

band,added at 8,kept at 12,lost at 12
top feature,,,
log_benes,285,1615,80
log_pay,5601,17764,2668
markup,3760,35497,2934
srvcs_per_bene,25230,46098,14622


band,added at 8,kept at 12,lost at 12
top feature,,,
log_benes,30.56,12.55,44.87
log_pay,1.04,1.52,1.07
markup,1.08,1.16,1.08
srvcs_per_bene,1.30,1.39,1.38


## 3. Combining the feature scores (flag %)

In [11]:
rules = {
    "any 1 feature > 3.5": flag_rule(p, one=3.5, two=3.5, n_min=99),
    "2+ features > 3.5": flag_rule(p, one=np.inf, two=3.5, n_min=2),
    "3+ features > 3.5": flag_rule(p, one=np.inf, two=3.5, n_min=3),
    "ours: 1 > 10 or 2+ > 3.5": default,
}
pd.Series({name: f.mean() * 100 for name, f in rules.items()}, name="flag %").round(2)

any 1 feature > 3.5         5.91
2+ features > 3.5           0.16
3+ features > 3.5           0.00
ours: 1 > 10 or 2+ > 3.5    1.24
Name: flag %, dtype: float64